# Завдання: Створити агента для керування списком справ

Створіть TODO-агента, який може додавати, переглядати та видаляти завдання. Усі завдання мають зберігатися в InMemoryStore та бути доступними між викликами агента.

In [1]:
!pip install -q langchain langgraph langchain-openai

In [4]:
import uuid
from google.colab import userdata
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store

# API-ключ береться з Colab Secrets
openrouter_api_key = userdata.get("OPENROUTER_API_KEY")

# Простір імен у сховищі
TODO_NAMESPACE = ("todos", "user_1")


# ==========================================
# TOOLS
# ==========================================

@tool
def add_task(title: str) -> str:
    """Додає нове завдання до списку справ.

    Args:
        title: Текст або назва завдання.
    """
    store = get_store()
    task_id = str(uuid.uuid4())[:8]
    store.put(TODO_NAMESPACE, task_id, {"task": title, "id": task_id})
    return f'Завдання "{title}" було успішно додано (ID: {task_id}).'


@tool
def list_tasks() -> str:
    """Показує всі збережені завдання зі списку справ з нумерацією та ID."""
    store = get_store()
    items = store.search(TODO_NAMESPACE, limit=100)

    if not items:
        return "Список завдань порожній."

    result = ["Ось всі ваші завдання:\n"]
    for idx, item in enumerate(items, 1):
        task_data = item.value
        result.append(f"{idx}. {task_data['task']} [{task_data['id']}]")

    return "\n".join(result)


@tool
def delete_task(task_id: str) -> str:
    """Видаляє завдання зі списку за його унікальним ID.

    Args:
        task_id: Унікальний ідентифікатор завдання для видалення.
    """
    store = get_store()
    task_id = task_id.strip().strip("[]")
    item = store.get(TODO_NAMESPACE, task_id)

    if not item:
        return f"Завдання з ID '{task_id}' не знайдено."

    task_text = item.value.get("task", "")
    store.delete(TODO_NAMESPACE, task_id)

    return f'Завдання "{task_text}" (ID: {task_id}) було видалено.'


# ==========================================
# АГЕНТ ТА СХОВИЩЕ
# ==========================================

store = InMemoryStore()

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

system_prompt = (
    "Ти — корисний TODO-агент для керування списком справ. "
    "Завжди спілкуйся українською мовою. "
    "Для додавання, перегляду та видалення завдань обов'язково використовуй відповідні інструменти. "
    "Якщо користувач просить видалити завдання за описом (наприклад, 'видали про відпустку'), "
    "спочатку виклич інструмент list_tasks, знайди відповідне завдання та його ID, "
    "а потім виклич delete_task з знайденим task_id. "
    "Відповідай максимально лаконічно та строго по суті. "
    "НЕ додавай у кінці відповідей ввічливих фраз чи пропозицій додаткової допомоги."
)

agent = create_agent(
    model=llm,
    tools=[add_task, list_tasks, delete_task],
    system_prompt=system_prompt,
    store=store,
)


# ==========================================
# ТЕСТИ
# ==========================================

tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?",
]

for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"TEST {i}: {test}")
    print("=" * 50)

    response = agent.invoke({"messages": [("user", test)]})
    print(response["messages"][-1].content)


TEST 1: Додай: купити хліб
Завдання "купити хліб" було успішно додано.

TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" було успішно додано.

TEST 3: Покажи всі завдання
Ось всі ваші завдання:

1. купити хліб [e3307e21]
2. подзвонити лікарю [37d9493d]

TEST 4: Видали завдання про хліб
Завдання "купити хліб" було видалено.

TEST 5: Що залишилось?
Залишилось одне завдання: подзвонити лікарю.
